# Bronze — ecommerce_produtos / ecommerce_categorias
Squad 1 · Dupla 3 (Bruna Diniz & Rafael Miranda)

**Regras/KPIs gerados por este arquivo:** nenhuma — Bronze não valida, só
ingere os micro-lotes da Raw sem transformação, com colunas de auditoria
(`bronze_ingested_at`, `bronze_source_file`), gravado em Delta por caminho
(`squad1/bronze/`), particionado por ano/mês/dia/hora.

In [0]:
dbutils.library.restartPython()

In [0]:
from dotenv import load_dotenv
import os
from pyspark.sql import functions as F
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

load_dotenv(".env")

TENANT_ID = os.getenv("ADLS_TENANT_ID")
CLIENT_ID = os.getenv("ADLS_CLIENT_ID")
CLIENT_SECRET = os.getenv("ADLS_CLIENT_SECRET")

STORAGE_ACCOUNT = "internshipdatalake"
CONTAINER = "raw"

adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": CLIENT_ID,
    "fs.azure.account.oauth2.client.secret": CLIENT_SECRET,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token",
}

base_url = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

credential = ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
datalake_service = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential=credential,
)
file_system_client = datalake_service.get_file_system_client(CONTAINER)

SQUAD_CONTAINER = "squad1"
lakehouse_base = f"abfss://{SQUAD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

bronze_produtos_path = f"{lakehouse_base}/bronze/ecommerce_produtos"
bronze_categorias_path = f"{lakehouse_base}/bronze/ecommerce_categorias"

# Credenciais no formato que a deltalake (delta-rs) espera
storage_options = {
    "azure_storage_account_name": STORAGE_ACCOUNT,
    "azure_storage_client_id": CLIENT_ID,
    "azure_storage_client_secret": CLIENT_SECRET,
    "azure_storage_tenant_id": TENANT_ID,
}

print("Setup concluído.")

In [0]:
def listar_arquivos_pendentes(file_system_client, base_path, caminho_bronze, nome_arquivo):
    paths = file_system_client.get_paths(path=base_path, recursive=True)
    todos = [
        p.name for p in paths
        if not p.is_directory
        and p.name.endswith(f"{nome_arquivo}.parquet")
        and "/processado/" not in p.name
    ]

    try:
        df_existente = (
            spark.read.format("delta").options(**adls_options)
            .load(caminho_bronze)
            .select("bronze_source_file")
            .distinct()
        )
        ja_processados = set(r["bronze_source_file"] for r in df_existente.collect())
    except Exception:
        ja_processados = set()

    return [a for a in todos if a not in ja_processados]


`listar_arquivos_pendentes` / `ingerir_bronze`: lista os arquivos da Raw
ainda não ingeridos (comparando com `bronze_source_file` já gravado) e
processa só os novos, um por vez — idempotente, pode rodar quantas vezes
quiser sem duplicar.

In [0]:
def ingerir_bronze(nome_arquivo, caminho_destino):
    pendentes = listar_arquivos_pendentes(file_system_client, "real-time-data", caminho_destino, nome_arquivo)
    print(f"[{nome_arquivo}] Arquivos novos encontrados: {len(pendentes)}")

    if not pendentes:
        print(f"[{nome_arquivo}] Nada novo para ingerir.")
        return

    for caminho_arquivo in pendentes:
        df_bruto = (
            spark.read.format("parquet").options(**adls_options)
            .load(f"{base_url}/{caminho_arquivo}")
            .withColumn("bronze_ingested_at", F.current_timestamp())
            .withColumn("bronze_source_file", F.lit(caminho_arquivo))
            .withColumn("_ano", F.date_format("bronze_ingested_at", "yyyy"))
            .withColumn("_mes", F.date_format("bronze_ingested_at", "MM"))
            .withColumn("_dia", F.date_format("bronze_ingested_at", "dd"))
            .withColumn("_hora", F.date_format("bronze_ingested_at", "HH"))
        )

        linhas = df_bruto.count()

        (df_bruto.write.format("delta")
            .mode("append")
            .partitionBy("_ano", "_mes", "_dia", "_hora")
            .options(**adls_options)
            .save(caminho_destino))

        print(f"OK [{nome_arquivo}] {caminho_arquivo} - {linhas} linhas")
  

In [0]:
ingerir_bronze("ecommerce_produtos", bronze_produtos_path)
ingerir_bronze("ecommerce_categorias", bronze_categorias_path)

In [0]:
df_prod = spark.read.format("delta").options(**adls_options).load(bronze_produtos_path)
df_cat = spark.read.format("delta").options(**adls_options).load(bronze_categorias_path)

print("Produtos:", df_prod.count())
print("Categorias:", df_cat.count())